# Exploring some basics from the MCS Zooms
## Written by Eric Rohr

In [ ]:
### import modules
import illustris_python as il # type: ignore
import matplotlib.pyplot as plt 
import numpy as np 
import matplotlib as mpl 
import matplotlib.cm as cm 
import matplotlib.patheffects as pe 
import matplotlib.transforms as transforms  
from matplotlib.gridspec import GridSpec  
import matplotlib.gridspec as gridspec  
from matplotlib.patches import Patch  
import matplotlib.patches as patches  
from mpl_toolkits.axes_grid1.inset_locator import inset_axes  
from mpl_toolkits.axes_grid1 import make_axes_locatable  
from scipy.ndimage import gaussian_filter  
from scipy import ndimage  
from scipy.interpolate import interp1d  
from scipy import interpolate  
from temet.util.sphMap import sphMap  #type: ignore
import scipy.stats  
from scipy.stats import norm  
from sklearn.neighbors import KernelDensity  
from scipy.stats import ks_2samp, anderson_ksamp  
from scipy.optimize import curve_fit  
from astropy.cosmology import Cosmology, FlatLambdaCDM, z_at_value
from astropy import units as u 
from astropy import constants as const 
import os
import glob
import csv
from pathlib import Path
import time
import h5py  
import rohr_utils as ru 
import utils.io as io
from utils.units import *
import random
import six  
import scida
from scida import load
import pint 
from createMCSTFiles import createMCSTFiles
import createOffsets
from stellar_array_helpers import expand_all_arrays

%matplotlib inline

plt.style.use('fullpage.mplstyle')

os.chdir('/u/reric/Scripts/')
! pwd



In [ ]:
class Sim: 
    """Create a class for the given simulation"""

    def __init__(self, simFamily, simName):
        """ initialize the class with basic info from the simulation"""

        kwargs = locals().copy()
        for _key in kwargs:
            if _key != 'self':
                setattr(self, _key, kwargs[_key])

        self.basePath = io.prepareSim(self.simFamily, self.simName)
        self.snapTimes = io.loadSnapTimes(self.basePath)
        self.Header = io.loadHeader(self.basePath, self.snapTimes['SnapNum'][0])
        self.Parameters = io.loadParameters(self.basePath, self.snapTimes['SnapNum'][0])
        self.Config = io.loadConfig(self.basePath, self.snapTimes['SnapNum'][0])
        self.snapNum_z0 = io.findSnapNum(self.basePath, 'Redshift', 0.0)

        # define short titles for plotting
        add_kwargs(self)


plot_kwargs = dict(marker='o', fillstyle='none', ms=3, mew=1.0, alpha=0.5)
med_kwargs = dict(marker='None', ls='-', lw=3, path_effects=[pe.Stroke(linewidth=4, foreground='white'), pe.Normal()], zorder=3)
hist_kwargs = dict(lw=0.2, alpha=0.4, ls='-')
percentiles_kwargs = dict(alpha=0.2)

def add_kwargs(sim, **kwargs):
    """ add kwargs to class"""

    if (sim.simFamily == 'IllustrisTNG'):
        c = 'tab:blue'
        label =  sim.simName[:-2]
    elif sim.simFamily == 'Eagle':
        c = 'fuchsia'
        label = 'Eagle'
    elif sim.simFamily == 'Simba':
        c = 'tab:orange'
        label = 'Simba'
    elif sim.simFamily == 'Illustris':
        c = 'tab:olive'
        label = 'Illustris'

    kwargs['c'] = kwargs['color'] = c
    kwargs['label'] = label

    sim.kwargs = kwargs
    return



In [ ]:
inputs = dict(IllustrisTNG=['TNG100-1'],
              Eagle=['Eagle100-1'],
              Illustris=['Illustris-1'],
              Simba=['Simba100-1'])

Sims = {}

for simFamily in inputs:
    simNames = inputs[simFamily]
    for simName in simNames:
        print(simFamily, simName)
        Sims[simName] = Sim(simFamily, simName)

In [ ]:
halo_fields = ['Group_M_Crit200', 'GroupMassType', 'GroupFirstSub']
subhalo_fields = ['SubhaloMassInRadType', 'SubhaloSFRinHalfRad', 'SubhaloGrNr']
for simName in Sims:
    sim = Sims[simName]
    sim.Halos_z0 = il.groupcat.loadHalos(sim.basePath, sim.snapNum_z0, halo_fields)
    io.convertGroupUnits(sim.basePath, sim.snapNum_z0, sim.Halos_z0)
    maskHalosCentrals_z0 = sim.Halos_z0['GroupFirstSub'] >= 0
    Subhalos = il.groupcat.loadSubhalos(sim.basePath, sim.snapNum_z0, subhalo_fields)
    r = {}
    for key in subhalo_fields:
        r[key] = Subhalos[key][(sim.Halos_z0['GroupFirstSub'][maskHalosCentrals_z0]).astype(np.int32)]
    r['count'] = sim.Halos_z0['GroupFirstSub'][maskHalosCentrals_z0].size
    io.convertGroupUnits(sim.basePath, sim.snapNum_z0, r)
    sim.Centrals_z0 = r
    sim.maskHalosCentals_z0 = maskHalosCentrals_z0

In [ ]:
savefig = False
outdirec = '../Figures/CosmoSimComparisons'
if not os.path.isdir(outdirec):
    os.makedirs(outdirec)

In [ ]:
# let's plot Mstar vs M200c at z=0 for all central galaxies and their host halos

def smoothCurve(ar, type='Gaussian', type_kwargs=dict(sigma=1)):
    """
    Smooth the given array by interpolating and applying a type of filter.
    Currently only supported for type == 'Gaussian', with a default sigma=1.
    """

    if not isinstance(ar, np.ndarray):
        ar = np.array(ar)
        if ar.size <= 1:
            return
    
    if type == 'Gaussian':
        return gaussian_filter(ar, **type_kwargs)
    else:
        raise ValueError('type %s not currently supported.'%type)
    

fig, ax = plt.subplots()

x_min = 10.**(10.5) * u.M_sun
binwidth = 0.2 # dex
flag_smoothCurve = True

for sim_i, simName in enumerate(Sims):
    
    sim = Sims[simName]
    add_kwargs(sim)
    _x = sim.Halos_z0['Group_M_Crit200'][sim.maskHalosCentals_z0]
    x_mask = _x > x_min
    x = _x[x_mask]
    _y = sim.Centrals_z0['SubhaloMassInRadType'][x_mask,4] / x
    y_mask = _y > 0
    y = _y[y_mask]
    x = x[y_mask]

    r = ru.return2dhiststats_dict(np.log10(x.value), np.log10(y.value), binwidth, percentiles=[16, 50, 84])
    if flag_smoothCurve:
        for key in r:
            r[key] = smoothCurve(r[key])

    #ax.plot(x[mask], y[mask] / y[mask], marker='.', ls='None', ms=1, alpha=0.2)
    ax.plot(10.**(r['bin_cents']), 10.**(r[50]), c=sim.kwargs['c'], **med_kwargs, label=sim.kwargs['label'] + ' (%d)'%(x.size))
    ax.fill_between(10.**(r['bin_cents']), 10.**(r[16]), 10.**(r[84]), color=sim.kwargs['c'], **percentiles_kwargs)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(10.**(10.5), 10.**(14.5))
ax.legend(title=r'Centrals at $z=0$')
ax.set_xlabel(r'Halo Mass $[M_{\rm 200c} / \rm{M_\odot}]$')
ax.set_ylabel(r'Stellar to Halo Mass Ratio $[M_\star / M_{\rm 200c}]$')

if savefig:
    fname = 'SHMR_z0.pdf'
    fig.savefig(os.path.join(outdirec, fname), bbox_inches='tight')

## begin re-developing createSGRP and associated codes to run for all cosmo sims

In [ ]:
sim = Sim('IllustrisTNG', 'TNG100-3')
centrals_flag = True


In [ ]:
centrals_flag = True
basePath = sim.basePath
snapNum_z0 = sim.snapNum_z0
fname = 'CosmoSimComparison_subfind.hdf5'
fpath = os.path.join(Path(basePath).parent, 'postprocessing', 'subfindGRP', fname)

subfindGRP = h5py.File(fpath, 'a')
SubfindIDs_z0 = subfindGRP['SubfindID'][:,0].astype(int)

In [ ]:

scalar_keys = ['SubhaloColdGasMass', 'SubhaloGasMass', 'SubhaloHotGasMass']
threed_keys = ['radii', 'vol_shells',
               'SubhaloColdGasMassShells', 'SubhaloColdGasDensityShells',
               'SubhaloHotGasMassShells', 'SubhaloHotGasDensityShells',
               'SubhaloGasMassShells', 'SubhaloDensityShells']

centrals_vector_keys = ['CGMTemperaturesHistogram', 'CGMTemperaturesHistogramBincents']
centrals_scalar_keys = ['SubhaloColdGasMassOutflowRate1.0R200c', 'SubhaloColdGasMassInflowRate1.0R200c',
                        'SubhaloHotGasMassOutflowRate1.0R200c', 'SubhaloHotGasMassInflowRate1.0R200c',
                        'SubhaloColdGasMassOutflowRate0.15R200c', 'SubhaloColdGasMassInflowRate0.15R200c',
                        'SubhaloHotGasMassOutflowRate0.15R200c', 'SubhaloHotGasMassInflowRate0.15R200c',]

In [ ]:
snapNum = snapNum_z0
subfindID = SubfindIDs_z0[0]
gas_ptn = il.util.partTypeNum('gas')
ptn = gas_ptn

subhalo = il.groupcat.loadSingle(basePath, snapNum, subhaloID=subfindID)
halo = il.groupcat.loadSingle(basePath, snapNum, haloID=subhalo['SubhaloGrNr'])
io.convertGroupUnits(basePath, snapNum, subhalo)
io.convertGroupUnits(basePath, snapNum, halo)
Header = io.loadHeader(basePath, snapNum)


In [ ]:
def loadGasCells(basePath, snapNum, subhaloID, fields=None, extra_fields=None):
    """ 
    load the gas cells using loadHaloWithoutSatellites, convert to physical units,
    add extra fields as necessary, and return the gas cells as a dictionary
    """
    gas_ptn = il.util.partTypeNum('Gas')
    gas = io.loadHaloWithoutSatellites(basePath, snapNum, subhaloID=subhaloID, ptn=gas_ptn)
    io.convertSnapshotUnits(basePath, snapNum, gas)
    io.computeTemperature(gas)
    io.computeCoolingTime(gas)
    io.computeGasPressure(gas)
    io.computeCellSizes(gas)
    io.computeEntropy(gas)
    subhalo = il.groupcat.loadSingle(basePath, snapNum, subhaloID=subhaloID)
    io.convertGroupUnits(basePath, snapNum, subhalo)
    io.computeRadii(gas, subhalo['SubhaloPos'], basePath=basePath, snapNum=snapNum)

    return gas



In [ ]:
dic = loadGasCells(basePath, snapNum, subfindID)

In [ ]:
def computeRadialProfile(dic, bins, weights=None):
    """ 
    Given the dictionary of convert snapshot units and the relevant bins (with units),
    compute either the density radial profile or the mass weighted radial profile. 
    Optionally center the profiles using center.
    Returns dens_shells (or weighted quantity), mass_shells, and vol_shells.
    """

    dens_shells = np.zeros(bins.size - 1, dtype=float) - 1.
    mass_shells = dens_shells.copy()
    vol_shells = dens_shells.copy()

    # check if 'Masses' is already present (not for dm)
    if 'Masses' not in dic:
        dic['Masses'] = np.ones(dic['count'], dtype=float) * (Header['MassTable'][ptn] * code_mass / Header['HubbleParam']).to(standard_mass)

    vol_shells = (4./3.) * np.pi * ((bins[1:])**3 - (bins[:-1])**3)
    mass_shells = np.histogram(dic['Radii'], bins=bins, weights=dic['Masses'])[0]

    if not weights:
        dens_shells = mass_shells / vol_shells
    else:
        weight_shells = np.histogram(dic['Radii'], bins=bins, weights=dic['Masses'] * dic[weights])[0]
        mask = weight_shells > 0
        dens_shells[mask] = weight_shells[mask] / mass_shells[mask]

    return dens_shells, mass_shells, vol_shells



In [ ]:
rmin_norm = 1.0e-3 # r / Rvir
rmax_norm = 1.0e1 # r / Rvir
radii_binwidth = 0.1 # r / Rvir, log
radii_bins_norm, radii_bincents_norm = ru.returnlogbins([rmin_norm, rmax_norm], radii_binwidth)
radii_bins_norm = np.insert(radii_bins_norm, 0, 0.)
radii_bincents_norm = np.insert(radii_bincents_norm, 0, radii_bins_norm[1]/2.)

r200c = halo['Group_R_Crit200']
radii_bins = radii_bins_norm * r200c
radii = radii_bincents_norm * r200c

dic = loadGasCells(basePath, snapNum, subfindID)

dens_shells, _, _ = computeRadialProfile(dic, radii_bins)
temp_shells = computeRadialProfile(dic, radii_bins, weights='Temperature')[0]

In [ ]:

fig, ax = plt.subplots()
ax.plot(radii, temp_shells, marker='None', ls='-')
ax.set_xscale('log')
ax.set_yscale('log')
